In [ ]:
# Cell 1: Setup, paths, and LIAR data loading
# Responsibility: prepare folders, load LIAR, and create three risk labels.

import os
import gc
import re
import json
import random
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup


# Main project paths
project_root = "/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1"
liar_data_dir = os.path.join(project_root, "data", "LIAR")

model_copy_dir = os.path.join(project_root, "model_training", "model_copy")
experiment_dir = os.path.join(project_root, "model_training", "claim_risk_bert_3class")
results_dir = os.path.join(project_root, "model_training", "claim_risk_results")

os.makedirs(model_copy_dir, exist_ok=True)
os.makedirs(experiment_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Basic experiment settings
random_seed = 42
base_model_name = "bert-base-uncased"
max_sequence_length = 128
batch_size = 16
max_epochs = 4

learning_rates = [1e-5, 2e-5, 3e-5]

# If Colab crashes, keep this as False first.
# Change to True only when you want to continue from saved last checkpoints.
resume_from_checkpoint = False

# If you already finished a run, this avoids training it again.
skip_finished_runs = True

risk_label_names = ["low_risk", "medium_risk", "high_risk"]

# LIAR original labels are truthfulness labels.
# Here they are used as a proxy for language risk.
risk_label_map = {
    "true": 0,
    "mostly-true": 0,
    "half-true": 1,
    "barely-true": 1,
    "false": 2,
    "pants-fire": 2,
}

def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

liar_column_names = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "context"
]

def load_liar_split(file_name):
    file_path = os.path.join(liar_data_dir, file_name)
    dataframe = pd.read_csv(file_path, sep="\t", header=None, names=liar_column_names)

    dataframe = dataframe[["label", "statement", "speaker", "context"]].copy()
    dataframe["statement"] = dataframe["statement"].astype(str)
    dataframe["risk_label"] = dataframe["label"].map(risk_label_map)

    dataframe = dataframe.dropna(subset=["risk_label"]).copy()
    dataframe["risk_label"] = dataframe["risk_label"].astype(int)

    return dataframe

train_df = load_liar_split("train.tsv")
validation_df = load_liar_split("valid.tsv")
test_df = load_liar_split("test.tsv")

def make_split_summary(dataframe, split_name):
    counts = dataframe["risk_label"].value_counts().sort_index()
    return {
        "split": split_name,
        "samples": len(dataframe),
        "low_risk": int(counts.get(0, 0)),
        "medium_risk": int(counts.get(1, 0)),
        "high_risk": int(counts.get(2, 0)),
    }

split_summary_df = pd.DataFrame([
    make_split_summary(train_df, "train"),
    make_split_summary(validation_df, "validation"),
    make_split_summary(test_df, "test"),
])

split_summary_path = os.path.join(results_dir, "table_01_dataset_split_summary.csv")
split_summary_df.to_csv(split_summary_path, index=False)

print("Device:", device)
print("Project root:", project_root)
print("LIAR data folder:", liar_data_dir)
print("\nDataset split summary:")
display(split_summary_df)
print("Saved dataset summary to:", split_summary_path)


Device: cuda
Project root: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1
LIAR data folder: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/data/LIAR

Dataset split summary:


,split,samples,low_risk,medium_risk,high_risk
0,train,10240,3638,3768,2834
1,validation,1284,420,485,379
2,test,1267,449,477,341


Saved dataset summary to: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_01_dataset_split_summary.csv


In [ ]:
# Cell 2: Dataset class and training helper functions
# Responsibility: keep the training loop simple and reusable.

tokenizer = BertTokenizer.from_pretrained(base_model_name)

class ClaimRiskDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.statements = dataframe["statement"].astype(str).tolist()
        self.labels = dataframe["risk_label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.statements)

    def __getitem__(self, index):
        encoded_text = self.tokenizer(
            self.statements[index],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {}
        item["input_ids"] = encoded_text["input_ids"].squeeze(0)
        item["attention_mask"] = encoded_text["attention_mask"].squeeze(0)
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)

        return item

def make_dataloader(dataframe, shuffle_rows):
    dataset = ClaimRiskDataset(dataframe, tokenizer, max_sequence_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_rows)

train_loader = make_dataloader(train_df, shuffle_rows=True)
validation_loader = make_dataloader(validation_df, shuffle_rows=False)
test_loader = make_dataloader(test_df, shuffle_rows=False)

def build_model():
    model = BertForSequenceClassification.from_pretrained(base_model_name, num_labels=3)
    model.to(device)
    return model

def make_weighted_loss(train_dataframe):
    class_counts = train_dataframe["risk_label"].value_counts().sort_index()
    class_weights = len(train_dataframe) / (len(risk_label_names) * class_counts)
    class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float).to(device)

    print("Class weights:", class_weights.round(3).to_dict())
    return torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

def evaluate_model(model, data_loader):
    model.eval()

    true_labels = []
    predicted_labels = []
    predicted_probabilities = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probabilities = torch.softmax(outputs.logits, dim=1)
            predictions = torch.argmax(probabilities, dim=1)

            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(predictions.cpu().numpy())
            predicted_probabilities.extend(probabilities.cpu().numpy())

    true_labels = np.array(true_labels)
    predicted_labels = np.array(predicted_labels)
    predicted_probabilities = np.array(predicted_probabilities)

    metrics = {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(true_labels, predicted_labels, average="macro"),
    }

    return true_labels, predicted_labels, predicted_probabilities, metrics

def train_one_epoch(model, train_loader, optimizer, scheduler, loss_function, epoch_number):
    model.train()

    total_loss = 0.0

    for batch_index, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_function(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (batch_index + 1) % 100 == 0:
            print("  epoch", epoch_number, "batch", batch_index + 1, "loss", round(loss.item(), 4))

    average_loss = total_loss / len(train_loader)
    return average_loss

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Helper functions are ready.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Helper functions are ready.


In [ ]:
# Cell 3: Experiment 01 - train several BERT models and choose the best validation run
# Responsibility: run a small learning-rate search without manually changing values.

loss_function = make_weighted_loss(train_df)

all_epoch_rows = []
run_summary_rows = []

selected_model_path = os.path.join(model_copy_dir, "claim_risk_bert_3class_selected_model.pt")
model_selection_csv_path = os.path.join(results_dir, "table_02_validation_model_selection.csv")
training_history_csv_path = os.path.join(results_dir, "table_03_training_history_by_epoch.csv")

for learning_rate in learning_rates:
    run_name = "bert_lr_" + str(learning_rate).replace("-", "minus").replace(".", "_")
    run_dir = os.path.join(experiment_dir, run_name)

    os.makedirs(run_dir, exist_ok=True)

    run_best_model_path = os.path.join(run_dir, "best_model.pt")
    run_last_checkpoint_path = os.path.join(run_dir, "last_checkpoint.pt")
    run_summary_path = os.path.join(run_dir, "run_summary.json")

    print("\n==================================================")
    print("Starting run:", run_name)
    print("Learning rate:", learning_rate)

    if skip_finished_runs and os.path.exists(run_summary_path):
        print("This run already has a summary. Loading it instead of retraining.")
        with open(run_summary_path, "r") as summary_file:
            saved_summary = json.load(summary_file)

        run_summary_rows.append(saved_summary)
        continue

    set_random_seed(random_seed)
    model = build_model()

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    total_training_steps = len(train_loader) * max_epochs
    warmup_steps = int(total_training_steps * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps
    )

    start_epoch = 0
    best_validation_macro_f1 = -1.0
    best_epoch = 0
    best_validation_accuracy = 0.0

    if resume_from_checkpoint and os.path.exists(run_last_checkpoint_path):
        print("Resuming from last checkpoint:", run_last_checkpoint_path)
        checkpoint = torch.load(run_last_checkpoint_path, map_location=device)

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        start_epoch = checkpoint["epoch"] + 1
        best_validation_macro_f1 = checkpoint["best_validation_macro_f1"]
        best_epoch = checkpoint["best_epoch"]
        best_validation_accuracy = checkpoint["best_validation_accuracy"]

    for epoch_index in range(start_epoch, max_epochs):
        epoch_number = epoch_index + 1
        print("\nRun:", run_name, "| Epoch:", epoch_number, "/", max_epochs)

        train_loss = train_one_epoch(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            loss_function=loss_function,
            epoch_number=epoch_number
        )

        validation_true_labels, validation_predictions, validation_probabilities, validation_metrics = evaluate_model(
            model,
            validation_loader
        )

        print("  train loss:", round(train_loss, 4))
        print("  validation accuracy:", round(validation_metrics["accuracy"], 4))
        print("  validation macro-F1:", round(validation_metrics["macro_f1"], 4))

        epoch_row = {
            "run_name": run_name,
            "learning_rate": learning_rate,
            "epoch": epoch_number,
            "train_loss": train_loss,
            "validation_accuracy": validation_metrics["accuracy"],
            "validation_macro_f1": validation_metrics["macro_f1"],
        }
        all_epoch_rows.append(epoch_row)

        pd.DataFrame(all_epoch_rows).to_csv(training_history_csv_path, index=False)

        if validation_metrics["macro_f1"] > best_validation_macro_f1:
            best_validation_macro_f1 = validation_metrics["macro_f1"]
            best_validation_accuracy = validation_metrics["accuracy"]
            best_epoch = epoch_number

            best_model_data = {
                "run_name": run_name,
                "base_model_name": base_model_name,
                "learning_rate": learning_rate,
                "epoch": epoch_number,
                "max_sequence_length": max_sequence_length,
                "risk_label_names": risk_label_names,
                "risk_label_map": risk_label_map,
                "validation_accuracy": best_validation_accuracy,
                "validation_macro_f1": best_validation_macro_f1,
                "model_state_dict": model.state_dict(),
            }

            torch.save(best_model_data, run_best_model_path)
            print("  saved new best model for this run.")

        last_checkpoint = {
            "run_name": run_name,
            "learning_rate": learning_rate,
            "epoch": epoch_index,
            "best_epoch": best_epoch,
            "best_validation_accuracy": best_validation_accuracy,
            "best_validation_macro_f1": best_validation_macro_f1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }

        torch.save(last_checkpoint, run_last_checkpoint_path)

        clear_memory()

    run_summary = {
        "run_name": run_name,
        "learning_rate": learning_rate,
        "best_epoch": best_epoch,
        "validation_accuracy": best_validation_accuracy,
        "validation_macro_f1": best_validation_macro_f1,
        "best_model_path": run_best_model_path,
    }

    with open(run_summary_path, "w") as summary_file:
        json.dump(run_summary, summary_file, indent=2)

    run_summary_rows.append(run_summary)

    del model
    clear_memory()

model_selection_df = pd.DataFrame(run_summary_rows)
model_selection_df = model_selection_df.sort_values("validation_macro_f1", ascending=False)
model_selection_df.to_csv(model_selection_csv_path, index=False)

best_run = model_selection_df.iloc[0].to_dict()
shutil.copy2(best_run["best_model_path"], selected_model_path)

print("\n==================================================")
print("Model selection finished.")
print("Best run:", best_run["run_name"])
print("Best learning rate:", best_run["learning_rate"])
print("Best epoch:", int(best_run["best_epoch"]))
print("Best validation macro-F1:", round(best_run["validation_macro_f1"], 4))
print("Selected model copied to:", selected_model_path)

display(model_selection_df)


Class weights: {0: 0.938, 1: 0.906, 2: 1.204}

Starting run: bert_lr_1eminus05
Learning rate: 1e-05


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...


Run: bert_lr_2eminus05 | Epoch: 1 / 4
  epoch 1 batch 100 loss 1.0968
  epoch 1 batch 200 loss 1.0601
  epoch 1 batch 300 loss 1.1332
  epoch 1 batch 400 loss 0.9857
  epoch 1 batch 500 loss 1.0356
  epoch 1 batch 600 loss 1.0479
  train loss: 1.0821
  validation accuracy: 0.4268
  validation macro-F1: 0.4103
  saved new best model for this run.

Run: bert_lr_2eminus05 | Epoch: 2 / 4
  epoch 2 batch 100 loss 1.0407
  epoch 2 batch 200 loss 1.1234
  epoch 2 batch 300 loss 0.9472
  epoch 2 batch 400 loss 0.9148
  epoch 2 batch 500 loss 1.0382
  epoch 2 batch 600 loss 1.0947
  train loss: 0.9948
  validation accuracy: 0.4883
  validation macro-F1: 0.4855
  saved new best model for this run.

Run: bert_lr_2eminus05 | Epoch: 3 / 4
  epoch 3 batch 100 loss 0.7682
  epoch 3 batch 200 loss 0.6722
  epoch 3 batch 300 loss 0.6001
  epoch 3 batch 400 loss 0.6739
  epoch 3 batch 500 loss 0.6471
  epoch 3 batch 600 loss 0.9742
  train loss: 0.8007
  validation accuracy: 0.4852
  validation macro-F

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Run: bert_lr_3eminus05 | Epoch: 1 / 4
  epoch 1 batch 100 loss 1.0911
  epoch 1 batch 200 loss 0.9926
  epoch 1 batch 300 loss 1.14
  epoch 1 batch 400 loss 0.9682
  epoch 1 batch 500 loss 1.0918
  epoch 1 batch 600 loss 1.0796
  train loss: 1.0792
  validation accuracy: 0.4245
  validation macro-F1: 0.4046
  saved new best model for this run.

Run: bert_lr_3eminus05 | Epoch: 2 / 4
  epoch 2 batch 100 loss 1.0578
  epoch 2 batch 200 loss 1.0349
  epoch 2 batch 300 loss 0.9375
  epoch 2 batch 400 loss 0.9145
  epoch 2 batch 500 loss 0.9965
  epoch 2 batch 600 loss 1.105
  train loss: 0.9898
  validation accuracy: 0.4751
  validation macro-F1: 0.4699
  saved new best model for this run.

Run: bert_lr_3eminus05 | Epoch: 3 / 4
  epoch 3 batch 100 loss 0.7576
  epoch 3 batch 200 loss 0.638
  epoch 3 batch 300 loss 0.4845
  epoch 3 batch 400 loss 0.7363
  epoch 3 batch 500 loss 0.6777
  epoch 3 batch 600 loss 0.9554
  train loss: 0.742
  validation accuracy: 0.4836
  validation macro-F1: 0.

,run_name,learning_rate,best_epoch,validation_accuracy,validation_macro_f1,best_model_path
0,bert_lr_1eminus05,0.00001,4,0.494548,0.492958,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
2,bert_lr_3eminus05,0.00003,4,0.494548,0.490470,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
1,bert_lr_2eminus05,0.00002,2,0.488318,0.485461,/content/drive/MyDrive/UoP/COMP3000/fake-news-...


In [ ]:
# Cell 4: Threshold tuning, ablation, final test, and report tables
# Responsibility: choose thresholds on validation, evaluate the final model on test, and save report-ready tables.

selected_checkpoint = torch.load(selected_model_path, map_location=device)

final_model = build_model()
final_model.load_state_dict(selected_checkpoint["model_state_dict"])
final_model.to(device)

def calculate_ordinal_risk_score(probability_table):
    # medium risk gets half weight, high risk gets full weight.
    return 0.5 * probability_table[:, 1] + probability_table[:, 2]

def convert_score_to_risk_label(risk_scores, medium_threshold, high_threshold):
    predicted_labels = []

    for risk_score in risk_scores:
        if risk_score >= high_threshold:
            predicted_labels.append(2)
        elif risk_score >= medium_threshold:
            predicted_labels.append(1)
        else:
            predicted_labels.append(0)

    return np.array(predicted_labels)

def detect_risk_signals(text):
    text_lower = str(text).lower()
    detected_signals = []

    sensational_words = ["shocking", "bombshell", "secret", "exposed", "disaster", "destroy"]
    vague_source_phrases = ["sources say", "experts say", "people are saying", "many believe", "some say"]
    absolutist_words = ["always", "never", "everyone", "nobody", "all of them", "completely"]
    conspiracy_phrases = ["cover up", "they do not want you to know", "mainstream media hides"]

    if any(word in text_lower for word in sensational_words):
        detected_signals.append("sensational_wording")

    if any(phrase in text_lower for phrase in vague_source_phrases):
        detected_signals.append("vague_source")

    if any(word in text_lower for word in absolutist_words):
        detected_signals.append("absolutist_wording")

    if any(phrase in text_lower for phrase in conspiracy_phrases):
        detected_signals.append("conspiracy_framing")

    if re.search(r"\b\d+(\.\d+)?\s*(%|percent|million|billion|times)\b", text_lower):
        detected_signals.append("specific_number_claim")

    return detected_signals

def calculate_rule_bonus(detected_signals):
    bonus = 0.0

    for signal in detected_signals:
        if signal == "conspiracy_framing":
            bonus += 0.10
        else:
            bonus += 0.05

    return min(bonus, 0.20)

# Validation evaluation for the selected model
validation_true_labels, validation_argmax_predictions, validation_probabilities, validation_metrics = evaluate_model(
    final_model,
    validation_loader
)

validation_risk_scores = calculate_ordinal_risk_score(validation_probabilities)

threshold_rows = []

medium_thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
high_thresholds = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

for medium_threshold in medium_thresholds:
    for high_threshold in high_thresholds:
        if medium_threshold >= high_threshold:
            continue

        threshold_predictions = convert_score_to_risk_label(
            validation_risk_scores,
            medium_threshold,
            high_threshold
        )

        threshold_rows.append({
            "medium_threshold": medium_threshold,
            "high_threshold": high_threshold,
            "validation_accuracy": accuracy_score(validation_true_labels, threshold_predictions),
            "validation_macro_f1": f1_score(validation_true_labels, threshold_predictions, average="macro"),
        })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df = threshold_df.sort_values("validation_macro_f1", ascending=False)

threshold_csv_path = os.path.join(results_dir, "table_04_threshold_tuning.csv")
threshold_df.to_csv(threshold_csv_path, index=False)

best_threshold_row = threshold_df.iloc[0]
best_medium_threshold = float(best_threshold_row["medium_threshold"])
best_high_threshold = float(best_threshold_row["high_threshold"])

print("Best thresholds from validation:")
print("Medium threshold:", best_medium_threshold)
print("High threshold:", best_high_threshold)
display(threshold_df.head(10))

# Ablation on validation
validation_rule_bonus = np.array([
    calculate_rule_bonus(detect_risk_signals(text))
    for text in validation_df["statement"]
])

validation_rule_scores = np.clip(validation_risk_scores + validation_rule_bonus, 0.0, 1.0)

validation_ordinal_predictions = convert_score_to_risk_label(
    validation_risk_scores,
    best_medium_threshold,
    best_high_threshold
)

validation_rule_predictions = convert_score_to_risk_label(
    validation_rule_scores,
    best_medium_threshold,
    best_high_threshold
)

ablation_rows = []

for method_name, method_predictions in [
    ("BERT argmax", validation_argmax_predictions),
    ("BERT ordinal thresholds", validation_ordinal_predictions),
    ("BERT ordinal plus rules", validation_rule_predictions),
]:
    ablation_rows.append({
        "method": method_name,
        "validation_accuracy": accuracy_score(validation_true_labels, method_predictions),
        "validation_macro_f1": f1_score(validation_true_labels, method_predictions, average="macro"),
    })

ablation_df = pd.DataFrame(ablation_rows)

ablation_csv_path = os.path.join(results_dir, "table_05_ablation_validation.csv")
ablation_df.to_csv(ablation_csv_path, index=False)

print("\nValidation ablation:")
display(ablation_df)

# Final test evaluation
test_true_labels, test_argmax_predictions, test_probabilities, test_argmax_metrics = evaluate_model(
    final_model,
    test_loader
)

test_risk_scores = calculate_ordinal_risk_score(test_probabilities)

test_rule_bonus = np.array([
    calculate_rule_bonus(detect_risk_signals(text))
    for text in test_df["statement"]
])

test_final_scores = np.clip(test_risk_scores + test_rule_bonus, 0.0, 1.0)

test_final_predictions = convert_score_to_risk_label(
    test_final_scores,
    best_medium_threshold,
    best_high_threshold
)

test_report_dict = classification_report(
    test_true_labels,
    test_final_predictions,
    target_names=risk_label_names,
    digits=4,
    output_dict=True
)

test_confusion_matrix = confusion_matrix(test_true_labels, test_final_predictions)

test_result_row = {
    "selected_run": selected_checkpoint["run_name"],
    "learning_rate": selected_checkpoint["learning_rate"],
    "best_epoch": selected_checkpoint["epoch"],
    "medium_threshold": best_medium_threshold,
    "high_threshold": best_high_threshold,
    "test_accuracy": accuracy_score(test_true_labels, test_final_predictions),
    "test_macro_f1": f1_score(test_true_labels, test_final_predictions, average="macro"),
    "low_risk_f1": test_report_dict["low_risk"]["f1-score"],
    "medium_risk_f1": test_report_dict["medium_risk"]["f1-score"],
    "high_risk_f1": test_report_dict["high_risk"]["f1-score"],
}

test_result_df = pd.DataFrame([test_result_row])

test_result_csv_path = os.path.join(results_dir, "table_06_final_test_result.csv")
test_report_json_path = os.path.join(results_dir, "claim_risk_final_test_report.json")
test_confusion_matrix_csv_path = os.path.join(results_dir, "table_07_final_confusion_matrix.csv")

test_result_df.to_csv(test_result_csv_path, index=False)

with open(test_report_json_path, "w") as report_file:
    json.dump(test_report_dict, report_file, indent=2)

pd.DataFrame(
    test_confusion_matrix,
    index=risk_label_names,
    columns=risk_label_names
).to_csv(test_confusion_matrix_csv_path)

print("\nFinal test result:")
display(test_result_df)

print("\nFinal classification report:")
print(classification_report(test_true_labels, test_final_predictions, target_names=risk_label_names, digits=4))

print("\nFinal confusion matrix:")
print(test_confusion_matrix)

# Reliability bins
test_confidence = test_probabilities.max(axis=1)
test_correct = (test_argmax_predictions == test_true_labels).astype(int)

reliability_df = pd.DataFrame({
    "confidence": test_confidence,
    "correct": test_correct,
})

reliability_df["confidence_bin"] = pd.cut(
    reliability_df["confidence"],
    bins=[0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
)

reliability_summary_df = reliability_df.groupby(
    "confidence_bin",
    observed=False
).agg(
    count=("correct", "size"),
    accuracy=("correct", "mean")
).reset_index()

reliability_csv_path = os.path.join(results_dir, "table_08_reliability_bins.csv")
reliability_summary_df.to_csv(reliability_csv_path, index=False)

print("\nReliability bins:")
display(reliability_summary_df)

# Small stress test table for report discussion
stress_test_claims = [
    {
        "claim_type": "sensational vague claim",
        "claim": "Experts say the government is hiding the shocking truth about crime statistics."
    },
    {
        "claim_type": "neutral local claim",
        "claim": "The city council approved a new recycling policy on Tuesday."
    },
    {
        "claim_type": "specific number claim",
        "claim": "The unemployment rate increased by 4 percent last year."
    },
    {
        "claim_type": "absolutist claim",
        "claim": "Everyone knows this policy will completely destroy the economy."
    },
    {
        "claim_type": "plain political claim",
        "claim": "A politician said the programme cost more than originally planned."
    },
]

def predict_one_claim(claim_text):
    encoded_claim = tokenizer(
        claim_text,
        truncation=True,
        padding="max_length",
        max_length=max_sequence_length,
        return_tensors="pt"
    )

    encoded_claim = {
        key: value.to(device)
        for key, value in encoded_claim.items()
    }

    final_model.eval()

    with torch.no_grad():
        outputs = final_model(**encoded_claim)
        probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    bert_score = 0.5 * probabilities[1] + probabilities[2]
    signals = detect_risk_signals(claim_text)
    rule_bonus = calculate_rule_bonus(signals)
    final_score = min(1.0, bert_score + rule_bonus)

    predicted_label = convert_score_to_risk_label(
        np.array([final_score]),
        best_medium_threshold,
        best_high_threshold
    )[0]

    return {
        "bert_score": round(float(bert_score), 4),
        "rule_bonus": round(float(rule_bonus), 4),
        "final_score": round(float(final_score), 4),
        "predicted_risk": risk_label_names[int(predicted_label)],
        "rule_signals": ", ".join(signals) if signals else "none",
    }

stress_test_rows = []

for item in stress_test_claims:
    prediction = predict_one_claim(item["claim"])

    stress_test_rows.append({
        "claim_type": item["claim_type"],
        "claim": item["claim"],
        "bert_score": prediction["bert_score"],
        "rule_bonus": prediction["rule_bonus"],
        "final_score": prediction["final_score"],
        "predicted_risk": prediction["predicted_risk"],
        "rule_signals": prediction["rule_signals"],
    })

stress_test_df = pd.DataFrame(stress_test_rows)

stress_test_csv_path = os.path.join(results_dir, "table_09_stress_test_examples.csv")
stress_test_df.to_csv(stress_test_csv_path, index=False)

print("\nStress test examples:")
display(stress_test_df)

# Manifest for report writing
manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "task": "claim-level misleading language risk classification",
    "base_model": base_model_name,
    "dataset": "LIAR",
    "label_mapping": risk_label_map,
    "risk_label_names": risk_label_names,
    "learning_rates_tried": learning_rates,
    "max_epochs": max_epochs,
    "batch_size": batch_size,
    "max_sequence_length": max_sequence_length,
    "selected_run": selected_checkpoint["run_name"],
    "selected_learning_rate": selected_checkpoint["learning_rate"],
    "selected_epoch": selected_checkpoint["epoch"],
    "best_medium_threshold": best_medium_threshold,
    "best_high_threshold": best_high_threshold,
    "selected_model_path": selected_model_path,
    "results_dir": results_dir,
}

manifest_path = os.path.join(results_dir, "claim_risk_experiment_manifest.json")

with open(manifest_path, "w") as manifest_file:
    json.dump(manifest, manifest_file, indent=2)

print("\nSaved report files:")
print(split_summary_path)
print(model_selection_csv_path)
print(training_history_csv_path)
print(threshold_csv_path)
print(ablation_csv_path)
print(test_result_csv_path)
print(test_confusion_matrix_csv_path)
print(reliability_csv_path)
print(stress_test_csv_path)
print(manifest_path)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Best thresholds from validation:
Medium threshold: 0.3
High threshold: 0.6


,medium_threshold,high_threshold,validation_accuracy,validation_macro_f1
7,0.30,0.60,0.484424,0.482805
13,0.35,0.60,0.481308,0.481478
6,0.30,0.55,0.478972,0.480131
14,0.35,0.65,0.479751,0.476657
8,0.30,0.65,0.482866,0.476469
12,0.35,0.55,0.475857,0.476272
15,0.35,0.70,0.477414,0.468110
19,0.40,0.60,0.471184,0.467406
9,0.30,0.70,0.480530,0.466771
1,0.25,0.60,0.474299,0.464898



Validation ablation:


...


Final classification report:
              precision    recall  f1-score   support

    low_risk     0.5194    0.3875    0.4439       449
 medium_risk     0.3948    0.5115    0.4457       477
   high_risk     0.4395    0.4047    0.4214       341

    accuracy                         0.4388      1267
   macro avg     0.4512    0.4346    0.4370      1267
weighted avg     0.4510    0.4388    0.4385      1267


Final confusion matrix:
[[174 224  51]
 [108 244 125]
 [ 53 150 138]]

Reliability bins:


,confidence_bin,count,accuracy
0,"(0.0, 0.5]",377,0.400531
1,"(0.5, 0.6]",408,0.450980
2,"(0.6, 0.7]",269,0.464684
3,"(0.7, 0.8]",153,0.660131
4,"(0.8, 0.9]",59,0.610169
5,"(0.9, 1.0]",1,1.000000



Stress test examples:


,claim_type,claim,bert_score,rule_bonus,final_score,predicted_risk,rule_signals
0,sensational vague claim,Experts say the government is hiding the shock...,0.6689,0.10,0.7689,high_risk,"sensational_wording, vague_source"
1,neutral local claim,The city council approved a new recycling poli...,0.5177,0.00,0.5177,medium_risk,none
2,specific number claim,The unemployment rate increased by 4 percent l...,0.3222,0.05,0.3722,medium_risk,specific_number_claim
3,absolutist claim,Everyone knows this policy will completely des...,0.6509,0.10,0.7509,high_risk,"sensational_wording, absolutist_wording"
4,plain political claim,A politician said the programme cost more than...,0.4194,0.00,0.4194,medium_risk,none



Saved report files:
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_01_dataset_split_summary.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_02_validation_model_selection.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_03_training_history_by_epoch.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_04_threshold_tuning.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_05_ablation_validation.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_06_final_test_result.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_07_final_confusion_matrix.csv
/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results

In [ ]:
# Cell 5: Baselines for three-risk classification
# Responsibility: compare the BERT model against simple non-text baselines.

def make_baseline_result_row(method_name, true_labels, predicted_labels):
    baseline_report = classification_report(
        true_labels,
        predicted_labels,
        target_names=risk_label_names,
        output_dict=True,
        zero_division=0
    )

    return {
        "method": method_name,
        "test_accuracy": accuracy_score(true_labels, predicted_labels),
        "test_macro_f1": f1_score(true_labels, predicted_labels, average="macro"),
        "low_risk_f1": baseline_report["low_risk"]["f1-score"],
        "medium_risk_f1": baseline_report["medium_risk"]["f1-score"],
        "high_risk_f1": baseline_report["high_risk"]["f1-score"],
    }

def make_random_baseline_result(method_name, class_probabilities, repeat_count=30):
    random_rows = []

    for repeat_index in range(repeat_count):
        random_generator = np.random.default_rng(random_seed + repeat_index)

        random_predictions = random_generator.choice(
            [0, 1, 2],
            size=len(test_true_labels),
            p=class_probabilities
        )

        random_rows.append(
            make_baseline_result_row(
                method_name,
                test_true_labels,
                random_predictions
            )
        )

    random_result_df = pd.DataFrame(random_rows)

    averaged_result = {
        "method": method_name,
    }

    for column_name in [
        "test_accuracy",
        "test_macro_f1",
        "low_risk_f1",
        "medium_risk_f1",
        "high_risk_f1",
    ]:
        averaged_result[column_name] = random_result_df[column_name].mean()
        averaged_result[column_name + "_std"] = random_result_df[column_name].std()

    return averaged_result

# Majority baseline: always predict the most common risk label in the training set.
majority_risk_label = int(train_df["risk_label"].value_counts().idxmax())
majority_predictions = np.full(len(test_true_labels), majority_risk_label)

# Uniform random baseline: each risk class has the same chance.
uniform_class_probabilities = np.array([1 / 3, 1 / 3, 1 / 3])

# Stratified random baseline: random guessing using the training label distribution.
train_risk_distribution = (
    train_df["risk_label"]
    .value_counts(normalize=True)
    .sort_index()
    .reindex([0, 1, 2])
    .fillna(0)
    .to_numpy()
)

baseline_rows = [
    make_baseline_result_row(
        "majority_class_baseline",
        test_true_labels,
        majority_predictions
    ),
    make_random_baseline_result(
        "uniform_random_baseline_30_runs",
        uniform_class_probabilities
    ),
    make_random_baseline_result(
        "stratified_random_baseline_30_runs",
        train_risk_distribution
    ),
]

baseline_df = pd.DataFrame(baseline_rows)

baseline_csv_path = os.path.join(
    results_dir,
    "table_13_risk_baselines.csv"
)

baseline_df.to_csv(baseline_csv_path, index=False)

print("Three-risk baselines:")
display(baseline_df)

print("Saved to:", baseline_csv_path)
print("Majority class:", risk_label_names[majority_risk_label])
print("Training risk distribution:", train_risk_distribution)


Three-risk baselines:


,method,test_accuracy,test_macro_f1,low_risk_f1,medium_risk_f1,high_risk_f1,test_accuracy_std,test_macro_f1_std,low_risk_f1_std,medium_risk_f1_std,high_risk_f1_std
0,majority_class_baseline,0.376480,0.182339,0.000000,0.547018,0.000000,NaN,NaN,NaN,NaN,NaN
1,uniform_random_baseline_30_runs,0.334254,0.332393,0.344303,0.355023,0.297853,0.015036,0.015119,0.021211,0.018976,0.021642
2,stratified_random_baseline_30_runs,0.339621,0.333530,0.357889,0.372772,0.269930,0.014423,0.014783,0.021323,0.016832,0.024191


Saved to: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_13_risk_baselines.csv
Majority class: medium_risk
Training risk distribution: [0.35527344 0.36796875 0.27675781]


In [ ]:
# Cell 6: Test ablation for final evaluation
# Responsibility: compare BERT argmax, ordinal score, and ordinal plus rules on the test set.

test_ablation_rows = []

test_ordinal_predictions = convert_score_to_risk_label(
    test_risk_scores,
    best_medium_threshold,
    best_high_threshold
)

test_rule_predictions = convert_score_to_risk_label(
    test_final_scores,
    best_medium_threshold,
    best_high_threshold
)

for method_name, method_predictions in [
    ("BERT argmax", test_argmax_predictions),
    ("BERT ordinal thresholds", test_ordinal_predictions),
    ("BERT ordinal plus rules", test_rule_predictions),
]:
    method_report = classification_report(
        test_true_labels,
        method_predictions,
        target_names=risk_label_names,
        output_dict=True,
        digits=4
    )

    test_ablation_rows.append({
        "method": method_name,
        "test_accuracy": accuracy_score(test_true_labels, method_predictions),
        "test_macro_f1": f1_score(test_true_labels, method_predictions, average="macro"),
        "low_risk_f1": method_report["low_risk"]["f1-score"],
        "medium_risk_f1": method_report["medium_risk"]["f1-score"],
        "high_risk_f1": method_report["high_risk"]["f1-score"],
    })

test_ablation_df = pd.DataFrame(test_ablation_rows)

test_ablation_csv_path = os.path.join(results_dir, "table_10_test_ablation.csv")
test_ablation_df.to_csv(test_ablation_csv_path, index=False)

print("Test ablation:")
display(test_ablation_df)
print("Saved to:", test_ablation_csv_path)

for method_name, method_predictions in [
    ("BERT argmax", test_argmax_predictions),
    ("BERT ordinal thresholds", test_ordinal_predictions),
    ("BERT ordinal plus rules", test_rule_predictions),
]:
    print("\n" + method_name)
    print(classification_report(test_true_labels, method_predictions, target_names=risk_label_names, digits=4))
    print(confusion_matrix(test_true_labels, method_predictions))


Test ablation:


,method,test_accuracy,test_macro_f1,low_risk_f1,medium_risk_f1,high_risk_f1
0,BERT argmax,0.471981,0.465831,0.524490,0.448759,0.424242
1,BERT ordinal thresholds,0.438043,0.436584,0.451852,0.439296,0.418605
2,BERT ordinal plus rules,0.438832,0.436971,0.443878,0.445662,0.421374


Saved to: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_10_test_ablation.csv

BERT argmax
              precision    recall  f1-score   support

    low_risk     0.4840    0.5724    0.5245       449
 medium_risk     0.4622    0.4361    0.4488       477
   high_risk     0.4650    0.3900    0.4242       341

    accuracy                         0.4720      1267
   macro avg     0.4704    0.4662    0.4658      1267
weighted avg     0.4707    0.4720    0.4690      1267

[[257 143  49]
 [165 208 104]
 [109  99 133]]

BERT ordinal thresholds
              precision    recall  f1-score   support

    low_risk     0.5069    0.4076    0.4519       449
 medium_risk     0.3937    0.4969    0.4393       477
   high_risk     0.4441    0.3959    0.4186       341

    accuracy                         0.4380      1267
   macro avg     0.4482    0.4334    0.4366      1267
weighted avg     0.4474    0.4380    0.4382      1267

[[183 217  49]
 [120 237 

In [ ]:
# Cell 6.1: Compare baselines with the selected BERT model
# Responsibility: create a report-ready table comparing simple baselines and trained BERT results.

best_bert_row = test_ablation_df.sort_values(
    "test_macro_f1",
    ascending=False
).iloc[0].copy()

model_vs_baseline_rows = []

for _, baseline_row in baseline_df.iterrows():
    model_vs_baseline_rows.append({
        "method": baseline_row["method"],
        "test_accuracy": baseline_row["test_accuracy"],
        "test_macro_f1": baseline_row["test_macro_f1"],
        "low_risk_f1": baseline_row["low_risk_f1"],
        "medium_risk_f1": baseline_row["medium_risk_f1"],
        "high_risk_f1": baseline_row["high_risk_f1"],
        "notes": "non-text baseline",
    })

model_vs_baseline_rows.append({
    "method": "selected_BERT_argmax",
    "test_accuracy": best_bert_row["test_accuracy"],
    "test_macro_f1": best_bert_row["test_macro_f1"],
    "low_risk_f1": best_bert_row["low_risk_f1"],
    "medium_risk_f1": best_bert_row["medium_risk_f1"],
    "high_risk_f1": best_bert_row["high_risk_f1"],
    "notes": "trained text model",
})

model_vs_baseline_df = pd.DataFrame(model_vs_baseline_rows)

model_vs_baseline_csv_path = os.path.join(
    results_dir,
    "table_14_model_vs_baselines.csv"
)

model_vs_baseline_df.to_csv(model_vs_baseline_csv_path, index=False)

print("Model vs baselines:")
display(model_vs_baseline_df)

print("Saved to:", model_vs_baseline_csv_path)


Model vs baselines:


,method,test_accuracy,test_macro_f1,low_risk_f1,medium_risk_f1,high_risk_f1,notes
0,majority_class_baseline,0.376480,0.182339,0.000000,0.547018,0.000000,non-text baseline
1,uniform_random_baseline_30_runs,0.334254,0.332393,0.344303,0.355023,0.297853,non-text baseline
2,stratified_random_baseline_30_runs,0.339621,0.333530,0.357889,0.372772,0.269930,non-text baseline
3,selected_BERT_argmax,0.471981,0.465831,0.524490,0.448759,0.424242,trained text model


Saved to: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_14_model_vs_baselines.csv


In [ ]:
# Cell 7: Experiment 02 - LoRA comparison with three random seeds
# Responsibility: compare full fine-tuning and LoRA using the selected configuration.

import time

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception:
    !pip install peft -q
    from peft import LoraConfig, get_peft_model, TaskType

comparison_learning_rate = float(selected_checkpoint["learning_rate"])
comparison_epochs = int(selected_checkpoint["epoch"])
comparison_seeds = [42, 123, 2025]

lora_results_csv_path = os.path.join(results_dir, "table_11_lora_seed_results.csv")
lora_summary_csv_path = os.path.join(results_dir, "table_12_lora_summary.csv")

def count_trainable_parameters(model):
    trainable_parameters = 0
    total_parameters = 0

    for parameter in model.parameters():
        parameter_count = parameter.numel()
        total_parameters += parameter_count

        if parameter.requires_grad:
            trainable_parameters += parameter_count

    return trainable_parameters, total_parameters

def build_lora_model():
    base_model = BertForSequenceClassification.from_pretrained(base_model_name, num_labels=3)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "value"],
        bias="none",
        modules_to_save=["classifier"]
    )

    lora_model = get_peft_model(base_model, lora_config)
    lora_model.to(device)

    return lora_model

def train_model_for_comparison(method_name, random_seed_value, learning_rate, number_of_epochs):
    set_random_seed(random_seed_value)

    if method_name == "full_fine_tuning":
        model = build_model()
    elif method_name == "lora":
        model = build_lora_model()
    else:
        raise ValueError("Unknown method name: " + method_name)

    trainable_parameters, total_parameters = count_trainable_parameters(model)

    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate
    )

    total_steps = len(train_loader) * number_of_epochs
    warmup_steps = int(total_steps * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    loss_function = make_weighted_loss(train_df)

    start_time = time.time()

    best_validation_macro_f1 = -1.0
    best_validation_accuracy = 0.0
    best_epoch_number = 0
    best_model_path = os.path.join(
        experiment_dir,
        method_name + "_seed_" + str(random_seed_value) + "_best_model.pt"
    )

    print("\n==================================================")
    print("Method:", method_name)
    print("Seed:", random_seed_value)
    print("Learning rate:", learning_rate)
    print("Epochs:", number_of_epochs)
    print("Trainable parameters:", trainable_parameters)
    print("Total parameters:", total_parameters)

    for epoch_index in range(number_of_epochs):
        epoch_number = epoch_index + 1

        print("\nMethod:", method_name, "| Seed:", random_seed_value, "| Epoch:", epoch_number)

        train_loss = train_one_epoch(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            loss_function=loss_function,
            epoch_number=epoch_number
        )

        validation_true_labels_now, validation_predictions_now, validation_probabilities_now, validation_metrics_now = evaluate_model(
            model,
            validation_loader
        )

        print("  train loss:", round(train_loss, 4))
        print("  validation accuracy:", round(validation_metrics_now["accuracy"], 4))
        print("  validation macro-F1:", round(validation_metrics_now["macro_f1"], 4))

        if validation_metrics_now["macro_f1"] > best_validation_macro_f1:
            best_validation_macro_f1 = validation_metrics_now["macro_f1"]
            best_validation_accuracy = validation_metrics_now["accuracy"]
            best_epoch_number = epoch_number

            torch.save({
                "method_name": method_name,
                "seed": random_seed_value,
                "learning_rate": learning_rate,
                "epoch": epoch_number,
                "model_state_dict": model.state_dict(),
                "base_model_name": base_model_name,
                "risk_label_names": risk_label_names,
                "max_sequence_length": max_sequence_length,
                "validation_accuracy": best_validation_accuracy,
                "validation_macro_f1": best_validation_macro_f1,
                "trainable_parameters": trainable_parameters,
                "total_parameters": total_parameters,
            }, best_model_path)

            print("  saved new best model for this method and seed.")

        clear_memory()

    training_time_seconds = time.time() - start_time

    best_checkpoint = torch.load(best_model_path, map_location=device)

    if method_name == "full_fine_tuning":
        best_model = build_model()
    else:
        best_model = build_lora_model()

    best_model.load_state_dict(best_checkpoint["model_state_dict"])
    best_model.to(device)

    test_true_labels_now, test_predictions_now, test_probabilities_now, test_metrics_now = evaluate_model(
        best_model,
        test_loader
    )

    test_report_now = classification_report(
        test_true_labels_now,
        test_predictions_now,
        target_names=risk_label_names,
        output_dict=True,
        digits=4
    )

    result_row = {
        "method": method_name,
        "seed": random_seed_value,
        "learning_rate": learning_rate,
        "best_epoch": best_epoch_number,
        "validation_accuracy": best_validation_accuracy,
        "validation_macro_f1": best_validation_macro_f1,
        "test_accuracy": test_metrics_now["accuracy"],
        "test_macro_f1": test_metrics_now["macro_f1"],
        "low_risk_f1": test_report_now["low_risk"]["f1-score"],
        "medium_risk_f1": test_report_now["medium_risk"]["f1-score"],
        "high_risk_f1": test_report_now["high_risk"]["f1-score"],
        "trainable_parameters": trainable_parameters,
        "total_parameters": total_parameters,
        "trainable_percent": trainable_parameters / total_parameters * 100,
        "training_time_seconds": training_time_seconds,
        "best_model_path": best_model_path,
    }

    del model
    del best_model
    clear_memory()

    return result_row

lora_comparison_rows = []

for method_name in ["full_fine_tuning", "lora"]:
    for seed_value in comparison_seeds:
        result_row = train_model_for_comparison(
            method_name=method_name,
            random_seed_value=seed_value,
            learning_rate=comparison_learning_rate,
            number_of_epochs=comparison_epochs
        )

        lora_comparison_rows.append(result_row)

        pd.DataFrame(lora_comparison_rows).to_csv(lora_results_csv_path, index=False)

lora_results_df = pd.DataFrame(lora_comparison_rows)
lora_results_df.to_csv(lora_results_csv_path, index=False)

lora_summary_df = lora_results_df.groupby("method").agg(
    seeds=("seed", "count"),
    mean_validation_macro_f1=("validation_macro_f1", "mean"),
    std_validation_macro_f1=("validation_macro_f1", "std"),
    mean_test_macro_f1=("test_macro_f1", "mean"),
    std_test_macro_f1=("test_macro_f1", "std"),
    mean_test_accuracy=("test_accuracy", "mean"),
    mean_high_risk_f1=("high_risk_f1", "mean"),
    mean_trainable_parameters=("trainable_parameters", "mean"),
    mean_trainable_percent=("trainable_percent", "mean"),
    mean_training_time_seconds=("training_time_seconds", "mean")
).reset_index()

lora_summary_df.to_csv(lora_summary_csv_path, index=False)

print("\nLoRA comparison seed-level results:")
display(lora_results_df)

print("\nLoRA comparison summary:")
display(lora_summary_df)

print("Saved seed-level results to:", lora_results_csv_path)
print("Saved summary to:", lora_summary_csv_path)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: {0: 0.938, 1: 0.906, 2: 1.204}

Method: full_fine_tuning
Seed: 42
Learning rate: 1e-05
Epochs: 4
Trainable parameters: 109484547
Total parameters: 109484547

Method: full_fine_tuning | Seed: 42 | Epoch: 1
  epoch 1 batch 100 loss 1.1072
  epoch 1 batch 200 loss 1.0203
  epoch 1 batch 300 loss 1.1382
  epoch 1 batch 400 loss 1.0058
  epoch 1 batch 500 loss 0.9774
  epoch 1 batch 600 loss 1.0341
  train loss: 1.0826
  validation accuracy: 0.4509
  validation macro-F1: 0.4414
  saved new best model for this method and seed.

Method: full_fine_tuning | Seed: 42 | Epoch: 2
  epoch 2 batch 100 loss 1.0851
  epoch 2 batch 200 loss 1.0566
  epoch 2 batch 300 loss 1.0123
  epoch 2 batch 400 loss 0.9925
  epoch 2 batch 500 loss 1.0537
  epoch 2 batch 600 loss 1.1604
  train loss: 1.0198
  validation accuracy: 0.4673
  validation macro-F1: 0.4671
  saved new best model for this method and seed.

Method: full_fine_tuning | Seed: 42 | Epoch: 3
  epoch 3 batch 100 loss 0.9928
  epoch 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...


LoRA comparison seed-level results:


,method,seed,learning_rate,best_epoch,validation_accuracy,validation_macro_f1,test_accuracy,test_macro_f1,low_risk_f1,medium_risk_f1,high_risk_f1,trainable_parameters,total_parameters,trainable_percent,training_time_seconds,best_model_path
0,full_fine_tuning,42,0.00001,4,0.494548,0.492958,0.471981,0.465831,0.524490,0.448759,0.424242,109484547,109484547,100.000000,1046.149583,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
1,full_fine_tuning,123,0.00001,4,0.485202,0.485504,0.484609,0.481733,0.546610,0.431555,0.467033,109484547,109484547,100.000000,1048.791808,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
2,full_fine_tuning,2025,0.00001,4,0.488318,0.489968,0.477506,0.472140,0.536278,0.452871,0.427273,109484547,109484547,100.000000,1046.072623,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
3,lora,42,0.00001,4,0.362150,0.358369,0.377269,0.368223,0.461692,0.327910,0.315068,297219,109781766,0.270736,747.808825,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
4,lora,123,0.00001,3,0.370717,0.370680,0.364641,0.364580,0.386174,0.356284,0.351282,297219,109781766,0.270736,746.457288,/content/drive/MyDrive/UoP/COMP3000/fake-news-...
5,lora,2025,0.00001,3,0.368380,0.360319,0.342541,0.338350,0.311050,0.388211,0.315789,297219,109781766,0.270736,746.298803,/content/drive/MyDrive/UoP/COMP3000/fake-news-...



LoRA comparison summary:


,method,seeds,mean_validation_macro_f1,std_validation_macro_f1,mean_test_macro_f1,std_test_macro_f1,mean_test_accuracy,mean_high_risk_f1,mean_trainable_parameters,mean_trainable_percent,mean_training_time_seconds
0,full_fine_tuning,3,0.489477,0.003751,0.473235,0.008007,0.478032,0.439516,109484547.0,100.000000,1047.004671
1,lora,3,0.363123,0.006617,0.357051,0.016298,0.361484,0.327380,297219.0,0.270736,746.854972


Saved seed-level results to: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_11_lora_seed_results.csv
Saved summary to: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/model_training/claim_risk_results/table_12_lora_summary.csv
